In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from daemon_analysis_tools.io.csv_handler import load_and_process_csv
from daemon_analysis_tools.io.yaml_handler import save_answers_to_yaml, load_answers_from_yaml
from daemon_analysis_tools.processing.grouper import group_questions_by_journal
from daemon_analysis_tools.services.discrepancy_resolver import resolve_discrepancy

Load and process data:
- Group answers by publisher and journal, trying to uniform names written in slightly different ways.
- Store in a DataFrame

In [3]:
data = load_and_process_csv("../../data/raw/rdp.csv")

Get a `dict` labeled by publisher names of `dict`s labeled by journal names of `dict`s of `Question` instances. The `.answer` attribute contains the answers given by the respondents and the explanations text to motivate it.

In [4]:
question_metadata_file = "../../data/metadata/question_metadata.yaml"

grouped_questions = group_questions_by_journal(data, question_metadata_file)

## Resolve discrepancies

The `Question` class has a `.resolve_discrepancies` method which updates `Question.anwsers` with the correct answer.

For example, let's consider IOP's 2D Materials. Question 7 has discrepancies.

In [5]:
for journal, data in grouped_questions["MDPI"].items():
    print(journal)
    for question, answer in data.items():
        if answer.has_discrepancies():
            answer.print_qa()
    print("\n\n")

actuators
7. Timing of data release
  Resp. 0:
    Answer: Required data must be available after an embargo period.
    Explanation: in the Recommended Data Availability Statements
  Resp. 1:
    Answer: Timing of data availability not adressed in RDP.
    Explanation: MDPI Research Data Policies

MDPI is committed to supporting open scientific exchange and enabling our authors to achieve best practices in sharing and archiving research data. We encourage all authors of articles published in MDPI journals to share their research data including, but not limited to protocols, analytic methods, raw data, processed data, code, software, algorithms, and study material. The data should be FAIR – findable, accessible, interoperable, and reusable – so that other researchers can locate and use the data.

We recommend that data and code should be deposited in a trusted repository that will allow for maximum reuse (see the Data Preservation section below). If this is not possible, authors are enc

Inconsistencies can be removed manually, passing the index of the correct respondent.

In [6]:
for j in ["actuators", "batteries", "catalysts", "coatings", "crystals", "marine_drugs", 
          "membranes", "molbank", "molecules", "photonics", "reactions", ]:
    for i in [7, 8, 9, 16, 18, 20]:
        resolve_discrepancy(
            grouped_questions["MDPI"][j][i],
            correct_answer=1,
            discrepancy_reason="Language understanding",
        )
    
    for i in [11, 12]:
        resolve_discrepancy(
            grouped_questions["MDPI"][j][i],
            correct_answer=1,
            discrepancy_reason="Text not found",
        )

for j in ["materials", "sensors"]:
    for i in [3, 4, 5, 8, 9, 16, 18, 20]:
        resolve_discrepancy(
            grouped_questions["MDPI"][j][i],
            correct_answer=1,
            discrepancy_reason="Language understanding",
        )
    
    for i in [7, 13, 14, 15]:
        resolve_discrepancy(
            grouped_questions["MDPI"][j][i],
            correct_answer=2,
            discrepancy_reason="Language understanding",
        )
    
    for i in [11, 12]:
        resolve_discrepancy(
            grouped_questions["MDPI"][j][i],
            correct_answer=2,
            discrepancy_reason="Text not found",
        )

    
for j in ["polymers"]:
    for i in [3, 4, 5, 8, 9]:
        resolve_discrepancy(
            grouped_questions["MDPI"][j][i],
            correct_answer=1,
            discrepancy_reason="Language understanding",
        )
    
    for i in [7, 13, 14, 15, 16, 18, 20]:
        resolve_discrepancy(
            grouped_questions["MDPI"][j][i],
            correct_answer=3,
            discrepancy_reason="Language understanding",
        )
    
    for i in [11, 12]:
        resolve_discrepancy(
            grouped_questions["MDPI"][j][i],
            correct_answer=3,
            discrepancy_reason="Text not found",
        )
    
    for i in [19]:
        resolve_discrepancy(
            grouped_questions["MDPI"][j][i],
            correct_answer=3,
            discrepancy_reason="Formatting",
        )

for j in ["nanomaterials"]:
    for i in [3, 4, 5, 8, 9]:
        resolve_discrepancy(
            grouped_questions["MDPI"][j][i],
            correct_answer=1,
            discrepancy_reason="Language understanding",
        )
    
    for i in [7, 13, 14, 15, 16, 18, 20]:
        resolve_discrepancy(
            grouped_questions["MDPI"][j][i],
            correct_answer=3,
            discrepancy_reason="Language understanding",
        )
    
    for i in [11, 12]:
        resolve_discrepancy(
            grouped_questions["MDPI"][j][i],
            correct_answer=3,
            discrepancy_reason="Text not found",
        )

In [7]:
for journal, data in grouped_questions["MDPI"].items():
    print("#############################################################")
    print(journal)
    for question, answer in data.items():
        if answer.has_discrepancies() and answer.correct_answer is None:
            answer.print_qa()

#############################################################
actuators
#############################################################
batteries
#############################################################
catalysts
#############################################################
coatings
#############################################################
crystals
#############################################################
marine_drugs
#############################################################
materials
#############################################################
membranes
#############################################################
molbank
#############################################################
molecules
#############################################################
nanomaterials
#############################################################
photonics
#############################################################
polymers
#############################################################


In [8]:
save_answers_to_yaml(
    grouped_questions,
    parent_folder="../../data/processed/all_answers",
    save_only=["MDPI"],
)

After doing this, the `.get_final_answer()` method returns the correct answer.